In [ ]:
from transformers import pipeline

model = pipeline("sentiment-analysis")
print(model("I love this project"))

In [ ]:
print(model("This is terrible"))
print(model("I'm not sure how I feel about this"))

In [ ]:
import requests

API_KEY = "b2a0d07341544104b699c5576a1bf7fd"

url = "https://newsapi.org/v2/everything"
params = {
    "q": "finance AND (stock market OR investment)",
    "language": "en",
    "apiKey": API_KEY
}

response = requests.get(url, params=params)
data = response.json()

articles = data["articles"]

news_list = []
for article in articles:
    text = article["title"] + " " + str(article["description"])
    news_list.append(text)

print(news_list[:3])

In [ ]:
# To get more articles

In [ ]:
import requests
from datetime import datetime, timedelta

API_KEY = "b2a0d07341544104b699c5576a1bf7fd"
url = "https://newsapi.org/v2/everything"

# Query parameters
query = "finance AND (stock market OR investment)"
language = "en"
page_size = 100  # max per request

# Date range setup: last 30 days
end_date = datetime.today()
start_date = end_date - timedelta(days=30)

all_articles = []

# Split the 30-day range into 3-day chunks (adjust as needed)
delta = timedelta(days=3)
current_start = start_date

while current_start < end_date:
    current_end = min(current_start + delta, end_date)
    
    page = 1
    while True:
        params = {
            "q": query,
            "language": language,
            "pageSize": page_size,
            "page": page,
            "from": current_start.strftime("%Y-%m-%d"),
            "to": current_end.strftime("%Y-%m-%d"),
            "apiKey": API_KEY
        }

        response = requests.get(url, params=params)
        data = response.json()

        if "articles" in data and data["articles"]:
            all_articles.extend(data["articles"])
            if len(data["articles"]) < page_size:
                break  # No more pages for this date range
            page += 1
        else:
            break  # No articles for this page/date range

    current_start += delta

# Extract text
news_list = []
for article in all_articles:
    text = article.get("title", "") + " " + str(article.get("description", ""))
    news_list.append(text)

print(f"Total articles fetched: {len(news_list)}")
print(news_list[:10])

In [ ]:
results = []

for news in news_list:
    result = model(news[:512])[0]
    results.append(result)

print(results[:7])

In [ ]:
import pandas as pd

df = pd.DataFrame(results)
print(df.head())

In [ ]:
import matplotlib.pyplot as plt

df['label'].value_counts().plot(kind='bar')
plt.title("News Sentiment Analysis")
plt.xlabel("Sentiment")
plt.ylabel("Count")
plt.show()